## 1.1 Data Upload

In [11]:
import pandas as pd

# Read NBE data
df = pd.read_csv('florida/NBEFloridaNEW.csv', dtype=str)

# Display first few rows of each to inspect
print("NBE shape:", df.shape)

# Preview datasets
df.head()


NBE shape: (211385, 21)


,YEAR,STATE_NAME,STRUCTURE_NUMBER,LATEST_STRUCTURE_NUMBER,OWNER_AGENCY,ELEMENT_CATEGORY,PARENT_ELEMENT_NUMBER,PARENT_ELEMENT_NAME,ELEMENT_NUMBER,ELEMENT_NAME,...,TOTAL_QUANTITY,CONDITION_STATE_1,CS_PERCENT_1,CONDITION_STATE_2,CS_PERCENT_2,CONDITION_STATE_3,CS_PERCENT_3,CONDITION_STATE_4,CS_PERCENT_4,HI
0,2020,12 - Florida,010001,010001,State Highway Agency,Culvert,NaN,NaN,241,Reinforced Concrete Culvert,...,366,0,0,323,88,43,11,0,0,63.01
1,2020,12 - Florida,010032,010032,State Highway Agency,Joint,NaN,NaN,301,Pourable Joint Seal,...,180,125,69,16,8,0,0,39,21,75.4
2,2020,12 - Florida,010032,010032,State Highway Agency,Wearing Surfaces and Protective Coatings,38,Reinforced Concrete Slab,510,Wearing Surfaces,...,4370,4280,97,0,0,90,2,0,0,98.62
3,2020,12 - Florida,010033,010033,State Highway Agency,Substructure,NaN,NaN,226,Prestressed Concrete Pile,...,6,5,83,0,0,1,16,0,0,88.83
4,2020,12 - Florida,010033,010033,State Highway Agency,Wearing Surfaces and Protective Coatings,38,Reinforced Concrete Slab,510,Wearing Surfaces,...,2716,2640,97,0,0,76,2,0,0,98.13


## 1.2 Encoding ELEMENT_CATEGORY + PARENT_ELEMENT_NAME + ELEMENT_NAME

In [12]:
# --- Encode ELEMENT_CATEGORY ---
cat_dummies = pd.get_dummies(df['ELEMENT_CATEGORY'], prefix='CAT')
cat_index = df.columns.get_loc("ELEMENT_CATEGORY")
for i, col in enumerate(cat_dummies.columns):
    df.insert(cat_index + 1 + i, col, cat_dummies[col])

# --- Encode PARENT_ELEMENT_NAME ---
parent_dummies = pd.get_dummies(df['PARENT_ELEMENT_NAME'], prefix='PARENT')
parent_index = df.columns.get_loc("PARENT_ELEMENT_NAME")
for i, col in enumerate(parent_dummies.columns):
    df.insert(parent_index + 1 + i, col, parent_dummies[col])

# --- Encode ELEMENT_NAME ---
elem_dummies = pd.get_dummies(df['ELEMENT_NAME'], prefix='ELEM')
elem_index = df.columns.get_loc("ELEMENT_NAME")
for i, col in enumerate(elem_dummies.columns):
    df.insert(elem_index + 1 + i, col, elem_dummies[col])



# Saving to file
df.to_csv("NBEFlorida_encoded.csv", index=False)


C:\Users\aksel\AppData\Local\Temp\ipykernel_9312\1403248234.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(elem_index + 1 + i, col, elem_dummies[col])
C:\Users\aksel\AppData\Local\Temp\ipykernel_9312\1403248234.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(elem_index + 1 + i, col, elem_dummies[col])
C:\Users\aksel\AppData\Local\Temp\ipykernel_9312\1403248234.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor 

## 1.3 Merging NBI to NBE according to Structure Number (and Year)

In [13]:
import pandas as pd

# --- Paths ---
nbe_paths = "florida/NBEFlorida_encoded.csv" 
nbi_path = "florida/NBIFloridaNEW.csv"

out_merged = "NEW_NBE_and_NBI_FL.csv"

# --- Read inputs ---
nbe = None
try:
    nbe = pd.read_csv(nbe_paths, dtype=str)
    print(f"Loaded NBE: {nbe_paths}")
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find NBE input at {nbe_paths}.")

nbi = pd.read_csv(nbi_path, dtype=str)

# --- Helpers ---
def norm_key(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    try:
        return str(int(float(s)))   # handles 5.55E+12 → 5550000000000
    except Exception:
        return s

def norm_year(s):
    try:
        return int(float(str(s).strip()))
    except Exception:
        return None

# --- Keys present in each dataset ---
nbe_key = "STRUCTURE_NUMBER" if "STRUCTURE_NUMBER" in nbe.columns else "LATEST_STRUCTURE_NUMBER"
nbi_key = "8 - Structure Number" if "8 - Structure Number" in nbi.columns else "2024 NBI Structure Number"

# --- Normalize join fields ---
nbe["_K_"] = nbe[nbe_key].map(norm_key)
nbe["_Y_"] = nbe["YEAR"].map(norm_year) if "YEAR" in nbe.columns else None

nbi["_K_"] = nbi[nbi_key].map(norm_key)
nbi["_Y_"] = nbi["Year"].map(norm_year) if "Year" in nbi.columns else None

# --- De-duplicate NBI per (key, year) ---
nbi_sorted = nbi.sort_values(by=["_K_", "_Y_"])
nbi_dedup = nbi_sorted.drop_duplicates(subset=["_K_", "_Y_"], keep="last").copy()

# --- Prefix NBI columns and merge on (key, year) ---
cols_to_prefix = [c for c in nbi_dedup.columns if c not in ["_K_", "_Y_"]]
nbi_pref = nbi_dedup[["_K_", "_Y_"] + cols_to_prefix] \
  .rename(columns={c: f"NBI_{c}" for c in cols_to_prefix})

merged = nbe.merge(nbi_pref, on=["_K_", "_Y_"], how="left", validate="m:1")


# --- Save final ---
merged.drop(columns=["_K_", "_Y_"]).to_csv(out_merged, index=False)

print("Saved:")
print(" -", out_merged)



Loaded NBE: florida/NBEFlorida_encoded.csv
Saved:
 - NEW_NBE_and_NBI_FL.csv


## 2.0 Data Exploration

In [3]:
import pandas as pd
nbe = pd.read_csv("oklahoma/NEW_NBE_and_NBI_OK.csv", dtype=str)
list(nbe)

['YEAR',
 'STATE_NAME',
 'STRUCTURE_NUMBER',
 'LATEST_STRUCTURE_NUMBER',
 'OWNER_AGENCY',
 'ELEMENT_CATEGORY',
 'CAT_Bearing',
 'CAT_Bridge Rail',
 'CAT_Culvert',
 'CAT_Deck/Slab',
 'CAT_Joint',
 'CAT_Substructure',
 'CAT_Superstructure',
 'CAT_Wearing Surfaces and Protective Coatings',
 'PARENT_ELEMENT_NUMBER',
 'PARENT_ELEMENT_NAME',
 'PARENT_Elastomeric Bearings',
 'PARENT_Other Bridge Rail',
 'PARENT_Prestressed Concrete Closed Web/Box Girder',
 'PARENT_Prestressed Concrete Girder/Beam',
 'PARENT_Prestressed Concrete Top Flange',
 'PARENT_Reinforced Concrete Abutment',
 'PARENT_Reinforced Concrete Bridge Rail',
 'PARENT_Reinforced Concrete Closed Web/Box Girder',
 'PARENT_Reinforced Concrete Column',
 'PARENT_Reinforced Concrete Culvert',
 'PARENT_Reinforced Concrete Deck',
 'PARENT_Reinforced Concrete Girder/Beam',
 'PARENT_Reinforced Concrete Pier Cap',
 'PARENT_Reinforced Concrete Slab',
 'PARENT_Steel Bridge Rail',
 'PARENT_Steel Closed Web/Box Girder',
 'PARENT_Steel Column',
